<a href="https://colab.research.google.com/github/jejefahdha/oilfield-relational-database-design/blob/main/2.%20notebook/volve_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import Libraries

In [3]:
import pandas as pd
datavolve = pd.read_excel('Volve production data.xlsx')


## 2. Load Dataset

The Volve production dataset is imported into Python for exploration and preprocessing.

In [2]:
from google.colab import files
uploaddata = files.upload()

Saving Volve production data.xlsx to Volve production data.xlsx


## 3. Exploratory Analysis

Before designing the database, the dataset was explored to understand its variables, identify missing values, and determine potential entities for normalization.

In [ ]:
datavolve

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15629,2016-09-14,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.07776,0.22879,0.01862,0.0,0.0,0.0,NaN,production,OP
15630,2016-09-15,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.08545,0.22914,0.00631,0.0,0.0,0.0,NaN,production,OP
15631,2016-09-16,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.08544,0.22896,0.01181,0.0,0.0,0.0,NaN,production,OP
15632,2016-09-17,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.07497,0.22846,0.02576,0.0,0.0,0.0,NaN,production,OP


In [ ]:
datavolve.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15634 entries, 0 to 15633
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   DATEPRD                   15634 non-null  datetime64[ns]
 1   WELL_BORE_CODE            15634 non-null  object        
 2   NPD_WELL_BORE_CODE        15634 non-null  int64         
 3   NPD_WELL_BORE_NAME        15634 non-null  object        
 4   NPD_FIELD_CODE            15634 non-null  int64         
 5   NPD_FIELD_NAME            15634 non-null  object        
 6   NPD_FACILITY_CODE         15634 non-null  int64         
 7   NPD_FACILITY_NAME         15634 non-null  object        
 8   ON_STREAM_HRS             15349 non-null  float64       
 9   AVG_DOWNHOLE_PRESSURE     8980 non-null   float64       
 10  AVG_DOWNHOLE_TEMPERATURE  8980 non-null   float64       
 11  AVG_DP_TUBING             8980 non-null   float64       
 12  AVG_ANNULUS_PRESS 

## 4. Entity Identification

The flat dataset contains both descriptive attributes (e.g., wells, facilities, dates) and transactional measurements (e.g., oil volume, gas volume, pressure). These were separated into dimension and fact tables following normalization principles.

Normalized tables:
1. dim_date: DATE_ID, DATE, YEAR, MONTH, DAY
2. dim facility: FACILITY_ID, NPD_FACILITY_CODE, NPD_FACILITY_NAME
3. dim_field: FIELD_ID, NPD_FIELD_CODE, NPD_FIELD_NAME
4. dim_well: WELL_ID, WELL_BORE_CODE, NPD_WELL_BORE_CODE, NPD_WELL_BORE_NAME, WELL_TYPE, FLOW_KIND
5. fact_daily_log: the rest of all variables

## 5. Create Dimension Tables
### a. Dimension Table: Date
A dimension table was created by extracting unique date information.

In [4]:
dim_date = (datavolve[["DATEPRD"]]
            .drop_duplicates()
            .sort_values("DATEPRD")
            .copy()
)

A surrogate key (DATE_ID) was generated to serve as the primary key.

In [5]:
dim_date["DATE_ID"] = (dim_date["DATEPRD"]
                      .dt.strftime("%Y%m%d")
                      .astype(int)
)

In [6]:
dim_date["YEAR"] = dim_date["DATEPRD"].dt.year
dim_date["MONTH"] = dim_date["DATEPRD"].dt.month
dim_date["DAY"] = dim_date["DATEPRD"].dt.day

In [7]:
dim_date = dim_date[["DATE_ID","DATEPRD","YEAR","MONTH","DAY"]]
dim_date

,DATE_ID,DATEPRD,YEAR,MONTH,DAY
9001,20070901,2007-09-01,2007,9,1
9002,20070902,2007-09-02,2007,9,2
9003,20070903,2007-09-03,2007,9,3
9004,20070904,2007-09-04,2007,9,4
9005,20070905,2007-09-05,2007,9,5
...,...,...,...,...,...
12323,20161005,2016-10-05,2016,10,5
12324,20161006,2016-10-06,2016,10,6
12325,20161007,2016-10-07,2016,10,7
12326,20161101,2016-11-01,2016,11,1


### b. Dimension Table: Facility

In [11]:
dim_facility = (datavolve[[
    "NPD_FACILITY_CODE",
    "NPD_FACILITY_NAME"]]
    .drop_duplicates()
    .sort_values("NPD_FACILITY_CODE")
    .reset_index(drop=True)
    .copy()
)


,FACILITY_ID,NPD_FACILITY_CODE,NPD_FACILITY_NAME
0,1,369304,MÆRSK INSPIRER


A surrogate key (FACILITY_ID) was generated to serve as the primary key.

In [ ]:
dim_facility["FACILITY_ID"]=dim_facility.index+1
dim_facility = dim_facility[["FACILITY_ID","NPD_FACILITY_CODE","NPD_FACILITY_NAME"]]
dim_facility

### c. Dimension Table: Field

In [10]:
dim_field = (datavolve[[
    "NPD_FIELD_CODE",
    "NPD_FIELD_NAME"]]
    .drop_duplicates()
    .sort_values("NPD_FIELD_CODE")
    .reset_index(drop=True)
    .copy()
)


,FIELD_ID,NPD_FIELD_CODE,NPD_FIELD_NAME
0,1,3420717,VOLVE


A surrogate key (FIELD_ID) was generated to serve as the primary key.

In [ ]:
dim_field["FIELD_ID"]=dim_field.index+1
dim_field = dim_field[["FIELD_ID","NPD_FIELD_CODE","NPD_FIELD_NAME"]]
dim_field

### d. Dimension Table: Well

In [8]:
dim_well = (datavolve[[
    "WELL_BORE_CODE",
    "NPD_WELL_BORE_CODE",
    "NPD_WELL_BORE_NAME",
    "WELL_TYPE",
    "FLOW_KIND"]]
    .drop_duplicates()
    .sort_values("NPD_WELL_BORE_CODE")
    .copy()
)
dim_well

,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,WELL_TYPE,FLOW_KIND
4967,NO 15/9-F-14 H,5351,15/9-F-14,OP,production
1911,NO 15/9-F-12 H,5599,15/9-F-12,OP,production
9001,NO 15/9-F-4 AH,5693,15/9-F-4,WI,injection
12328,NO 15/9-F-5 AH,5769,15/9-F-5,WI,injection
15474,NO 15/9-F-5 AH,5769,15/9-F-5,OP,production
15473,NO 15/9-F-5 AH,5769,15/9-F-5,WI,production
746,NO 15/9-F-11 H,7078,15/9-F-11,OP,production
8023,NO 15/9-F-15 D,7289,15/9-F-15 D,OP,production
1,NO 15/9-F-1 C,7405,15/9-F-1 C,OP,production
0,NO 15/9-F-1 C,7405,15/9-F-1 C,WI,production


Since WELL_TYPE is not well characteristic, for example for WELL 5769 consist of both of WELL_TYPE values ('OP' and 'WI'). So does the FLOW_KIND. That's why both of it better belong to fact_daily_log table.

Updated Normalized tables:
1. dim_date: DATE_ID, DATE, YEAR, MONTH, DAY
2. dim_field: FIELD_ID, NPD_FIELD_CODE, NPD_FIELD_NAME
3. dim facility: FACILITY_ID, NPD_FACILITY_CODE, NPD_FACILITY_NAME
4. dim_well: WELL_ID, WELL_BORE_CODE, NPD_WELL_BORE_CODE, NPD_WELL_BORE_NAME
5. fact_daily_log: the rest of all variables but group it as safety sensor first and then production

In [9]:
dim_well = (datavolve[[
    "WELL_BORE_CODE",
    "NPD_WELL_BORE_CODE",
    "NPD_WELL_BORE_NAME"]]
    .drop_duplicates()
    .sort_values("NPD_WELL_BORE_CODE")
    .reset_index(drop=True)
    .copy()
)

,WELL_ID,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME
0,1,NO 15/9-F-14 H,5351,15/9-F-14
1,2,NO 15/9-F-12 H,5599,15/9-F-12
2,3,NO 15/9-F-4 AH,5693,15/9-F-4
3,4,NO 15/9-F-5 AH,5769,15/9-F-5
4,5,NO 15/9-F-11 H,7078,15/9-F-11
5,6,NO 15/9-F-15 D,7289,15/9-F-15 D
6,7,NO 15/9-F-1 C,7405,15/9-F-1 C


A surrogate key (WELL_ID) was generated to serve as the primary key.

In [ ]:
dim_well["WELL_ID"]=dim_well.index+1
dim_well = dim_well[["WELL_ID","WELL_BORE_CODE","NPD_WELL_BORE_CODE","NPD_WELL_BORE_NAME"]]
dim_well

## 6. Create Fact Table

In [12]:
fact_daily_log = (datavolve[[
    "NPD_WELL_BORE_CODE",
    "DATEPRD",
    "NPD_FIELD_CODE",
    "NPD_FACILITY_CODE",
    "ON_STREAM_HRS",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
    "BORE_WI_VOL",
    "FLOW_KIND",
    "WELL_TYPE",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_DP_TUBING",
    "AVG_ANNULUS_PRESS",
    "AVG_CHOKE_SIZE_P",
    "AVG_WHP_P",
    "AVG_WHT_P",
    "DP_CHOKE_SIZE",
    "AVG_CHOKE_UOM"
    ]]
    .sort_values("DATEPRD")
    .reset_index(drop=True)
    .copy()
)

,LOG_ID,DATE_ID,WELL_ID,FIELD_ID,FACILITY_ID,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,...,WELL_TYPE,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,AVG_CHOKE_UOM
0,1,20070901,4,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,20070901,3,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,20070902,4,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,20070902,3,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,20070903,3,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15629,15630,20161005,3,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15630,15631,20161006,3,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15631,15632,20161007,3,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15632,15633,20161101,3,1,1,NaN,NaN,NaN,NaN,NaN,...,WI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Business keys from the original dataset (NPD_WELL_BORE_CODE, DATEPRD, NPD_FIELD_CODE, NPD_FACILITY_CODE) were replaced with surrogate keys by joining the corresponding dimension tables. These surrogate keys later become foreign keys in the relational database.

In [ ]:
fact_daily_log = fact_daily_log.merge(
      dim_well[["WELL_ID","NPD_WELL_BORE_CODE"]],
      on="NPD_WELL_BORE_CODE",
      how="left")

fact_daily_log = fact_daily_log.merge(
      dim_date[["DATE_ID","DATEPRD"]],
      on="DATEPRD",
      how="left")

fact_daily_log = fact_daily_log.merge(
      dim_field[["FIELD_ID","NPD_FIELD_CODE"]],
      on="NPD_FIELD_CODE",
      how="left")

fact_daily_log = fact_daily_log.merge(
      dim_facility[["FACILITY_ID","NPD_FACILITY_CODE"]],
      on="NPD_FACILITY_CODE",
      how="left")

fact_daily_log = fact_daily_log.drop(
      columns=["NPD_WELL_BORE_CODE",
               "DATEPRD",
               "NPD_FIELD_CODE",
               "NPD_FACILITY_CODE"])

A surrogate key (LOG_ID) was generated to serve as the primary key.

In [ ]:
fact_daily_log["LOG_ID"] = fact_daily_log.index + 1

fact_daily_log = fact_daily_log[[
    "LOG_ID",
    "DATE_ID",
    "WELL_ID",
    "FIELD_ID",
    "FACILITY_ID",
    "ON_STREAM_HRS",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
    "BORE_WI_VOL",
    "FLOW_KIND",
    "WELL_TYPE",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_DP_TUBING",
    "AVG_ANNULUS_PRESS",
    "AVG_CHOKE_SIZE_P",
    "AVG_WHP_P",
    "AVG_WHT_P",
    "DP_CHOKE_SIZE",
    "AVG_CHOKE_UOM"
]]

fact_daily_log

## 7. Export Tables
The normalized tables were exported as CSV files for implementation in SQL.


In [ ]:
dfs = {
    "dim_well": dim_well,
    "dim_date": dim_date,
    "dim_field": dim_field,
    "dim_facility": dim_facility,
    "fact_daily_log": fact_daily_log
}

for name, df in dfs.items():
    df.to_csv(f"{name}.csv", index=False)

from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/project/volve/"

for name, df in dfs.items():
    df.to_csv(f"{save_path}{name}.csv", index=False)

Mounted at /content/drive


## 8. Validation

Simple row count validation checks to compare with the ERD implemented in SQL.

In [13]:
len(dim_date)

3327

In [14]:
len(dim_facility)

1

In [15]:
len(dim_field)

1

In [16]:
len(dim_well)

7

In [17]:
len(fact_daily_log)

15634

In [18]:
len(datavolve)

15634